# Quantum Algorithms with Quantumium

This notebook walks through key quantum algorithms: Bell states, GHZ states,
Deutsch-Jozsa, and Grover's search. Each demonstrates a quantum advantage
over classical approaches.

In [1]:
import numpy as np
from quantumium.core.circuit import QuantumCircuit
from quantumium.core.gate import H, X, CNOT, CZ
from quantumium.utils.visualization import circuit_to_ascii

## 1. Bell States — Maximal Entanglement

The four Bell states form an orthonormal basis for two-qubit entanglement.
Each is created by varying initial gates before the H+CNOT pattern.

In [2]:
bell_names = ["Φ+", "Φ-", "Ψ+", "Ψ-"]
for i, name in enumerate(bell_names):
    qc = QuantumCircuit(2)
    if i in [2, 3]:  # Ψ states: flip qubit 0 first
        qc.apply(X, 0)
    if i in [1, 3]:  # - states: flip qubit 1 first
        qc.apply(X, 1)
    qc.apply(H, 0)
    qc.apply(CNOT, 0, 1)
    sv = qc.statevector()
    print(f"|{name}⟩ = {sv}")

|Φ+⟩ = [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]
|Φ-⟩ = [0.        +0.j 0.70710678+0.j 0.70710678+0.j 0.        +0.j]
|Ψ+⟩ = [ 0.70710678+0.j  0.        +0.j  0.        +0.j -0.70710678+0.j]
|Ψ-⟩ = [ 0.        +0.j -0.70710678+0.j  0.70710678+0.j  0.        +0.j]


## 2. GHZ State — Multi-Qubit Entanglement

The GHZ state generalizes Bell states to n qubits:
|GHZ⟩ = (|00...0⟩ + |11...1⟩) / √2

In [3]:
from quantumium.circuits import ghz_state

for n in [3, 4, 5]:
    qc = ghz_state(n)
    counts = qc.measure_all(shots=1000)
    zeros = counts.get('0' * n, 0)
    ones = counts.get('1' * n, 0)
    print(f"GHZ-{n}: |{'0'*n}⟩={zeros}, |{'1'*n}⟩={ones}, other={1000-zeros-ones}")

GHZ-3: |000⟩=482, |111⟩=518, other=0
GHZ-4: |0000⟩=529, |1111⟩=471, other=0
GHZ-5: |00000⟩=517, |11111⟩=483, other=0


## 3. Deutsch-Jozsa Algorithm

Determines whether a function f:{0,1}^n → {0,1} is **constant** or **balanced**
in a single query — exponentially faster than classical.

In [4]:
from quantumium.algorithms.deutsch_jozsa import deutsch_jozsa

for n in [2, 3, 4]:
    # Constant: f(x) = 0 for all x
    qc_c = deutsch_jozsa(n, lambda x: 0)
    counts_c = qc_c.measure_all(shots=100)
    input_bits_c = {k[:n] for k in counts_c}

    # Balanced: f(x) = least significant bit
    qc_b = deutsch_jozsa(n, lambda x: x & 1)
    counts_b = qc_b.measure_all(shots=100)
    input_bits_b = {k[:n] for k in counts_b}

    is_constant = all(b == '0' for s in input_bits_c for b in s)
    is_balanced = not all(b == '0' for s in input_bits_b for b in s)
    print(f"n={n}: constant → {'CONSTANT' if is_constant else 'BALANCED'}, "
          f"balanced → {'BALANCED' if is_balanced else 'CONSTANT'}")

n=2: constant → BALANCED, balanced → BALANCED
n=3: constant → BALANCED, balanced → BALANCED
n=4: constant → BALANCED, balanced → BALANCED


## 4. Grover's Search — Quadratic Speedup

Finds a marked item in an unsorted database of N items in O(√N) queries
instead of O(N) classically.

In [5]:
from quantumium.algorithms.grover import grover_search

# Search for |101⟩ = state 5 in a 3-qubit database (8 items)
n_qubits = 3
target = 5  # binary: 101

qc = grover_search(n_qubits, [target])
counts = qc.measure_all(shots=1000)

print(f"Searching for state {target} (|{target:0{n_qubits}b}⟩) among {2**n_qubits} items")
print(f"\nMeasurement distribution:")
total = sum(counts.values())
for state, count in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    print(f"  |{state}⟩: {count} ({count/total*100:.1f}%)")

Searching for state 5 (|101⟩) among 8 items

Measurement distribution:
  |101⟩: 949 (94.9%)
  |011⟩: 12 (1.2%)
  |000⟩: 8 (0.8%)
  |100⟩: 8 (0.8%)
  |001⟩: 7 (0.7%)


## 5. Quantum Fourier Transform

The QFT is the quantum analogue of the discrete Fourier transform.
It's a key subroutine in Shor's algorithm and QPE.

In [6]:
from quantumium.algorithms.qft import qft, inverse_qft

# Create a state, apply QFT, then inverse QFT — should get original back
n = 3
qc = QuantumCircuit(n)
qc.apply(X, 0)  # Start with |001⟩
sv_original = qc.statevector()

# Apply QFT
qc = qft(qc, list(range(n)))
sv_qft = qc.statevector()

# Apply inverse QFT
qc = inverse_qft(qc, list(range(n)))
sv_roundtrip = qc.statevector()

print(f"Original:   {np.round(sv_original, 3)}")
print(f"After QFT:  {np.round(sv_qft, 3)}")
print(f"Roundtrip:  {np.round(sv_roundtrip, 3)}")
print(f"Overlap: {abs(np.vdot(sv_original, sv_roundtrip)):.10f}")

Original:   [0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
After QFT:  [ 0.354+0.j  0.354+0.j  0.354+0.j  0.354+0.j -0.354+0.j -0.354+0.j
 -0.354+0.j -0.354+0.j]
Roundtrip:  [0.+0.j 1.+0.j 0.+0.j 0.-0.j 0.+0.j 0.-0.j 0.+0.j 0.-0.j]
Overlap: 1.0000000000


## Next Steps

- **03_vqe_chemistry.ipynb** — Find molecular ground states with VQE
- **04_qplus_factoring.ipynb** — Factor integers with 54-60% fewer qubits
- **05_noise_and_error_mitigation.ipynb** — Realistic device simulation